# IOAI — 2026 Local Roai Soil Quality (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train_data.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-soil.zip', '/tmp/soil.zip')
    with zipfile.ZipFile('/tmp/soil.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train_data.csv' if os.path.exists('data/train_data.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

In [ ]:
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

In [ ]:
root_path = "data"

seed = 42

# Data

In [ ]:
train_df = pd.read_csv(f"{root_path}/train_data.csv")
test_df = pd.read_csv(f"{root_path}/test_data.csv")

train_df.head()

In [ ]:
train_df.isna().sum().sort_values(ascending=False)

# Subtask 1

Nutrient_Index = 0.4 * Nitrogen + 0.3 * Phosphorus + 0.3 * Potassium.

In [ ]:
subtask1 = 0.4 * test_df["Nitrogen"] + 0.3 * test_df["Phosphorus"] + 0.3 * test_df["Potassium"]

# Subtask 2

- Acid dacă pH < 6.0
- Neutral dacă 6.0 ≤ pH ≤ 7.5
- Alkaline dacă pH > 7.5

In [ ]:
def cls_ph(ph):
    if ph < 6.0:
        return "Acid"
    if ph <= 7.5:
        return "Neutral"
    return "Alkaline"

subtask2 = test_df["pH"].apply(cls_ph)

In [ ]:
sns.histplot(subtask2)

# Subtask 3

In [ ]:
moistures = train_df["Moisture"][train_df["Moisture"].notna()]
train_moisture = moistures.median()

subtask3 = (test_df["Moisture"] > train_moisture).astype(int)

# Subtask 4

In [ ]:
train_counts = train_df["Soil_Type"].value_counts()

In [ ]:
subtask4 = test_df["Soil_Type"].map(train_counts)

# Subtask 5

In [ ]:
train_df.head()

In [ ]:
def prep_df(df: pd.DataFrame):
    y = df["Suitability"] if "Suitability" in df else None
    df = df.drop(["ID", "Suitability"], axis=1, errors='ignore')

    df["ph_class"] = df["pH"].apply(cls_ph) # most relevant
    df["index"] = 0.4 * df["Nitrogen"] + 0.3 * df["Phosphorus"] + 0.3 * df["Potassium"]

    missing_cols = df[df.isna()]
    for col in missing_cols:
        filler = df[col].mean() if df[col].dtype != 'object' else df[col].mode()
        df[col] = df[col].fillna(filler)

    cat_cols = df.select_dtypes(include='object').columns
    for col in cat_cols:
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop([col], axis=1)

    if y is not None:
        X = df
        return X, y
    return df

In [ ]:
X, y = prep_df(train_df)

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

In [ ]:
def evaluate(clf):
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    return f1_score(y_test, preds, pos_label="Unfavorable")

In [ ]:
lr = LogisticRegression(max_iter=5000)

evaluate(lr)

In [ ]:
clf = lr
clf.fit(X, y)

In [ ]:
X_test_real = prep_df(test_df)
subtask5 = clf.predict(X_test_real)

# Submission

In [ ]:
def build_subtask(sid, ans):
    return pd.DataFrame({
        "subtaskID": sid,
        "datapointID": test_df["ID"],
        "answer": ans
    })

subtasks = [
    (1, subtask1),
    (2, subtask2),
    (3, subtask3),
    (4, subtask4),
    (5, subtask5),
]

submission_df = pd.concat([build_subtask(sid, ans) for sid, ans in subtasks])

In [ ]:
submission_df.head()

In [ ]:
submission_df.to_csv(f"{root_path}/submission.csv", index=False)

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)